# Notebook 04: Supervised Learning and Assignment 3 Bridge

## Session Overview
This session starts with supervised learning using scikit-learn, then shifts into the data-analysis workflow students need for Assignment 3.

The earlier data-exploration case-study session is no longer part of the public sequence because presentations are already complete.

## Learning Goals

- Explain supervised learning in plain language.
- Split data into training and test sets.
- Train and evaluate simple classification models.
- Recognize common model-evaluation mistakes.
- Connect model practice to Assignment 3's dataset workflow.
- Load, clean, summarize, visualize, and cluster a tabular dataset.

## Warm-up Questions

1. What is something you might want a model to predict?
2. What is the difference between an input feature and an output target?
3. Why should we test a model on examples it did not train on?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import KMeans

# Part 1: Supervised Learning

Supervised learning uses examples with known answers.

- `X`: input features.
- `y`: target or answer.
- Classification predicts categories.
- Regression predicts numbers.

Today we focus on classification.

## Section 1: Load a Classification Dataset

In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")

iris_df = X.copy()
iris_df["species_name"] = y.map(dict(enumerate(iris.target_names)))
iris_df.head()


### Exercise 1.1: Name `X` and `y`

The model will predict flower species from measurements.

1. Which columns are the input features in `X`?
2. What does `y` store?
3. In plain English, what is one row of `X` paired with one value of `y`?


## Section 2: Train/Test Split

Train on one part of the data, then test on examples the model did not see during training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


**What does `stratify=y` mean?**

For a classification problem, `y` contains the class labels. `stratify=y` tells `train_test_split` to keep about the same class proportions in the training set and the test set.

Example: the iris dataset has three species. If each species is about one-third of the full dataset, stratifying helps the test set stay close to one-third of each species too. This makes the test set a fairer check of the model.


### Exercise 2.1: Explain the Train/Test Split Inputs

Look at the `train_test_split(...)` code above. Write one plain-English sentence for each input. For `stratify=y`, explain what class balance means.

- `X`
- `y`
- `test_size=0.25`
- `random_state=1`
- `stratify=y`


## Section 3: Model 1 - K-Nearest Neighbors

KNN predicts a class by looking at nearby training examples.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

knn_predictions = knn.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_predictions)

print("KNN accuracy:", knn_accuracy)

### Exercise 3.1: Try Different Values of `k`

Train KNN models with `k = 1`, `3`, `5`, and `9`. Store each test accuracy in a small results table.

In [ ]:
knn_results = []

for k in [1, 3, 5, 9]:
    # TODO: train a KNN model with this value of k
    # TODO: predict on X_test
    # TODO: append {"k": k, "accuracy": accuracy} to knn_results
    pass

pd.DataFrame(knn_results)

## Section 4: Model 2 - Decision Tree

Decision trees split the data using feature-based questions.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=1)
tree.fit(X_train, y_train)

tree_predictions = tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print("Decision tree accuracy:", tree_accuracy)

### Exercise 4.1: Compare the Two Models

Create a DataFrame with one row for KNN and one row for the decision tree. Include model name and test accuracy.

In [ ]:
model_results = []

# TODO: add KNN and decision tree results

pd.DataFrame(model_results)

## Section 5: Evaluation

Accuracy is useful, but it is not the whole story. A confusion matrix shows which classes are getting mixed up.

### Common Evaluation Metrics

Different questions need different metrics.

| Scenario | Useful metrics | What the metric helps answer |
| --- | --- | --- |
| Balanced classification | accuracy | What fraction of predictions were correct? |
| Imbalanced classification | precision, recall, F1 | Is the model missing rare cases or creating too many false alarms? |
| False positives are costly | precision | When the model predicts the positive class, how often is it right? |
| False negatives are costly | recall | Of all real positive cases, how many did the model find? |
| Probability or ranking task | ROC-AUC | Does the model rank positive examples above negative examples? |
| Numeric prediction | MAE, MSE, RMSE, R² | How far are predictions from the real numbers? |
| Clustering with no labels | inertia, silhouette score, plots | Are the groups compact, separated, and interpretable? |

A metric is useful only when you can explain what question it answers.


### Confusion Matrix Guide

A confusion matrix compares true labels with predicted labels.

- Rows are the true labels.
- Columns are the predicted labels.
- One cell reads as: true row class predicted as column class.
- The diagonal cells are correct predictions.
- Off-diagonal cells are mistakes.

For a binary problem, two common mistake types are:

- **False positive:** the model predicted positive, but the real answer was negative.
- **False negative:** the model predicted negative, but the real answer was positive.

Precision focuses on false positives. Recall focuses on false negatives.


In [ ]:
confusion = confusion_matrix(y_test, tree_predictions)
confusion_df = pd.DataFrame(
    confusion,
    index=iris.target_names,
    columns=iris.target_names,
)

confusion_df

In [ ]:
print(classification_report(
    y_test,
    tree_predictions,
    target_names=iris.target_names,
    zero_division=0,
))


### Exercise 5.1: Read Matrix and Choose a Metric

Use the confusion matrix and classification report above.

1. Name one confused pair of classes, or explain why there are no mistakes on this split.
2. If false positives were costly, which metric would you watch most closely?
3. If false negatives were costly, which metric would you watch most closely?
4. Write one careful sentence that starts with: `On this test split...`


# Part 2: Assignment 3 Bridge

Assignment 3 is not mainly about supervised learning. It asks students to work with a real tabular dataset: load it, clean it, summarize it, visualize it, and apply clustering.

This bridge section uses the same Kaggle motorcycle dataset students need for Assignment 3: `data/all_bikez_curated.csv`.

The cells below are intentionally scaffolds. They name the decision points and useful tools, but they do not complete the assignment for you.


## Section 6: Assignment 3 Workflow Map

Assignment 3 asks for this kind of workflow:

1. Load a CSV with pandas.
2. Inspect rows, columns, and data types.
3. Clean missing values or text-like numeric columns.
4. Select numeric columns.
5. Use `describe()` and `corr()`.
6. Filter rows with multiple conditions.
7. Make clear visualizations.
8. Apply K-means clustering to numeric data.
9. Interpret results cautiously.

## Section 7: Load the Assignment 3 Dataset

The real Assignment 3 dataset is already in the course `data/` folder.

Use `low_memory=False` so pandas reads mixed columns more consistently. Loading the file is setup; the analysis decisions still come from you.


In [ ]:
DATA_PATH = "data/all_bikez_curated.csv"

motorcycle_info_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Rows and columns:", motorcycle_info_raw.shape)
motorcycle_info_raw.head()

### Exercise 7.1: Inspect Before You Decide

Use inspection methods to answer these questions before writing analysis code:

1. How many rows and columns are in the raw dataset?
2. Which columns look numeric? Which columns are text/categorical?
3. Which columns have many missing values?
4. Which numeric-looking columns might need cleaning before analysis?

Useful tools: `head()`, `shape`, `columns`, `info()`, `isna().sum()`, `describe()`, and `corr()`.


In [ ]:
# TODO: inspect the first rows.
# TODO: inspect shape and column names.
# TODO: inspect data types and missing values.
# TODO: after selecting numeric columns, use describe() and corr().

print("Start from motorcycle_info_raw. What do you need to inspect first?")


## Section 8: Clean and Select Numeric Columns

Some columns may look numeric but arrive as text. Before changing data, inspect a few values and decide what cleaning is justified.

Questions to answer before code:

1. Which column needs conversion?
2. What values should become missing after conversion?
3. How much data would be removed if you drop missing values?
4. Which columns should be used for numeric summaries or clustering?


In [ ]:
motorcycle_info = motorcycle_info_raw.copy()
motorcycle_info_num = pd.DataFrame()

# TODO: inspect suspicious numeric-looking columns before converting them.
# Hint: check dtype and a few unique/sample values.

# TODO: if a column should be numeric, convert it with pd.to_numeric(..., errors="coerce").
# Hint: decide whether any string cleanup is needed before conversion.

# TODO: decide how to handle missing values after conversion.
# Hint: compare the shape before and after cleaning so you know how much data changed.

# TODO: after cleaning, create motorcycle_info_num from the numeric columns.

print("Raw shape:", motorcycle_info_raw.shape)
print("Current working shape:", motorcycle_info.shape)
print("Fill in the cleaning and numeric-column selection steps above.")


In [ ]:
# TODO: display summary statistics for numeric columns.
# TODO: display the correlation matrix for numeric columns.
# Hint: if the output looks strange, go back and inspect dtypes/missing values.

if motorcycle_info_num.empty:
    print("Select numeric columns first.")
else:
    print("Use describe() and corr() here after your cleaning decision.")


### Exercise 8.1: Filter with Multiple Conditions

Create one filtered subset using at least three conditions.

Suggested practice target: Yamaha sport motorcycles with displacement greater than 950 ccm.

Before coding, identify:

1. Which column stores brand?
2. Which column stores category?
3. Which column stores displacement?
4. Do any text comparisons need capitalization handling?

Reminder: use parentheses around each condition and combine them with `&`.


In [ ]:
filtered_motorcycles = None

# TODO: build one condition at a time.
# TODO: combine the conditions with &.
# TODO: store the filtered DataFrame in filtered_motorcycles.

filtered_motorcycles


### Exercise 8.2: Visualize Brand Counts

Create a horizontal bar chart of motorcycle counts by brand.

Guiding questions:

1. Should you plot every brand or only the most common brands?
2. Which pandas method counts categories?
3. Which chart direction makes long brand names readable?
4. What title and axis labels will make the plot clear?


In [ ]:
# TODO: count motorcycles by brand.
# TODO: choose how many top brands to display.
# TODO: make a horizontal bar chart with a title and axis labels.

brand_counts = None
brand_counts


## Section 9: K-means Preview

K-means is unsupervised learning: it groups rows without known labels. In Assignment 3, students apply it to numeric motorcycle features.

Do not treat the code as a formula to copy. The important decisions are which numeric columns to use, whether the data needs scaling, and how cautiously to interpret the resulting groups.


In [ ]:
candidate_cluster_columns = ["Displacement (ccm)", "Torque (Nm)", "Power (hp)", "Dry weight (kg)"]

# TODO: choose numeric columns for clustering.
# Hint: start with candidate_cluster_columns only if they exist and have enough non-missing values.
cluster_columns = []

# TODO: create cluster_data from the cleaned DataFrame.
# TODO: create and fit a KMeans model.
# TODO: attach the cluster labels to a copy of the rows used for clustering.
# TODO: inspect a few rows and summarize each cluster.

if not cluster_columns:
    print("Choose cluster_columns and complete the K-means steps above.")


### Exercise 9.1: Interpret Clusters Carefully

After you run K-means, choose one cluster and describe what its motorcycles seem to have in common.

Use evidence from summaries, not just the cluster number:

1. How many rows are in each cluster?
2. What are the average feature values in each cluster?
3. Which features seem to separate the clusters most?
4. What is one caution about over-interpreting the groups?


## Assignment 3 Checklist

Before submitting Assignment 3, check that your notebook includes:

- CSV loading code.
- `head()` and `info()` checks.
- missing-value handling.
- numeric-column selection.
- `describe()` and `corr()` output.
- at least one filtered result using multiple conditions.
- clearly labeled plots.
- K-means clustering on numeric data.
- short explanations of what each result means.

## Reflection Questions

1. What makes a task supervised rather than unsupervised?
2. Why do we keep test data separate?
3. Which Assignment 3 step feels most error-prone?
4. What should we avoid claiming from one model score or one cluster plot?